# Quack-Weave core and native architecture

Source snapshot `23d96d45b94c4673bf202a21718d7e6c93130117`. The native crate assembles the extension, installs DuckDB's C++ graph grammar and optimizer hooks, and implements the graph engine adapters. The core crate owns graph query binding, registration identity, scan-route selection, and the typed batch contract. DuckDB owns SQL evaluation and decides when to request graph rows.

Start with the crate topology, then follow a read from SQL lowering through a provider session. The type inventory identifies the Rust handoffs. The decision and lifecycle diagrams explain the routes, replacement races, and evidence flow from a completed scan. The [Ladybug cloud storage execution design](#ladybug-cloud-storage-execution-design) describes how the native adapter queries remote graph snapshots.

The executable diagrams cover all 13 Mermaid headers with an implemented profile in the notebook's NS-Mermaid registry. Solver results apply to the declared topology, guards, traces, and numeric inputs. The source links and SQLLogic cases cover the compiled implementation. The appendix records the registry status of the other headers.

## Build and registration topology

The native `rusty_quack` crate builds a `cdylib` and exports `quack_weave` through DuckDB C API v2.0.1. Its entrypoint first calls core `register` for the portable template functions. It then creates the enabled `GraphProvider` values, calls core `register_query` with a `GqlParser` function pointer, and installs the native grammar using the returned registry. The normal build includes Ladybug, RelateDB and Neo4j. The native `template-compat` entrypoint and the WASM entrypoint call only the portable registration path.

The link dependency runs from native to core. At runtime, the core calls native engine implementations through the `ReadBackend`, `ReadSession` and `BatchReader` traits. The native C++ code also calls the registered core table functions by name. The arrows below show those runtime calls and declared zone crossings.

[Native entrypoint](../crates/native/src/lib.rs) · [Native Cargo features](../crates/native/Cargo.toml) · [Core portable registration](../crates/core/src/lib.rs) · [WASM entrypoint](../crates/wasm/src/lib.rs)

In [ ]:
architecture-beta
    group host(cloud)[DuckDB host]
    group native(cloud)[crates native]
    group core(cloud)[crates core]
    group engines(cloud)[Read engines]
    group wasm(cloud)[WASM target]
    service sql(server)[Parser Planner and Operators] in host
    service entry(server)[Native Extension Entry] in native
    service grammar(server)[C++ Grammar and Optimizer Hooks] in native
    service adapters(server)[Rust Graph Adapters] in native
    service template(server)[Portable Functions] in core
    service registry(database)[ProviderRegistry and ConfigSnapshot] in core
    service query(server)[GraphQuery and ScanProjection] in core
    service bridge(server)[Scan and Typed Batch Bridge] in core
    service lady(database)[Ladybug] in engines
    service relate(database)[RelateDB] in engines
    service neo(database)[Neo4j Query API] in engines
    service wasm_entry(server)[Rusty Quack WASM Entry] in wasm
    sql:R --> L:entry
    sql:B --> T:grammar
    entry:R --> L:template
    entry:B --> T:registry
    grammar:R --> L:query
    query:T --> B:registry
    query:R --> L:adapters
    query:B --> T:bridge
    adapters:T --> B:lady
    adapters:R --> L:relate
    adapters:B --> T:neo
    wasm_entry:R --> L:template
    %% @ns-zone-allow host native
    %% @ns-zone-allow native core
    %% @ns-zone-allow core native
    %% @ns-zone-allow native engines
    %% @ns-zone-allow wasm core

### Provider boundary and data ownership

| Provider | Evidence at bind | Native execution and delivery | Rewrite policy |
|---|---|---|---|
| Ladybug | Local observed catalog and available table populations, or the catalog of the attached cloud snapshot | Read-only prepared Cypher through Ladybug FFI. The reader owns the connection and result, decodes Arrow or graph values, and constructs typed batches. | Local files support label-catalog rewrites and the opt-in ordered runtime-key prefix. Cloud registrations use the host SQL route. |
| RelateDB | Observed catalog when available, or a declared schema for binding | Opens a read connection for the registered bundle. The adapter calls `query_with_params` and drains its eager row result into batches. | Catalog-backed rewrites require observed types. The ordered prefix stays in DuckDB. |
| Neo4j | An explicit schema or declared catalog for the supported graph fragment | Builds a remote session, resolves an approved credential, and sends the Query API request over HTTPS. The adapter reads and decodes a capped response before yielding batches. | The descriptor disables PGQ rewrites. DuckDB can still evaluate inference-supported reads and their residual SQL. |

The local registry validates external access and the file identity. The remote path validates DuckDB's remote permission, an approved origin, a credential reference, and request limits before submission. S3 and GCS snapshots use Ladybug's embedded HTTPFS attachment path. Core rejects Azure registrations before any object request. The [cloud execution design](#ladybug-cloud-storage-execution-design) records the provider-specific qualification gates. A batch-size budget limits bridge output. Provider-internal work and eager intermediates can exceed the size of a batch.

[Ladybug adapter](../crates/native/src/ladybug.rs) · [RelateDB adapter](../crates/native/src/relate_backend.rs) · [Neo4j adapter](../crates/native/src/neo4j_backend.rs) · [Remote config and limits](../crates/core/src/qw_query/remote.rs)

In [ ]:
architecture-beta
    group runtime(cloud)[DuckDB Process]
    group data(cloud)[Local Graph Files]
    group storage(cloud)[Cloud Graph Snapshots]
    group remote(cloud)[Remote Graph Endpoint]
    service query(server)[Core Admission and Scan] in runtime
    service ladybug(server)[Native Ladybug FFI and Cypher] in runtime
    service file_reader(server)[Native RelateDB Adapter] in runtime
    service http_reader(server)[Native Neo4j Adapter] in runtime
    service secret(server)[DuckDB Secret Resolver] in runtime
    service httpfs(server)[Embedded HTTPFS Read Only ATTACH] in runtime
    service azure_io(server)[Azure Registration Rejected] in runtime
    service lady(database)[Local Ladybug lbdb] in data
    service relate(database)[RelateDB Bundle] in data
    service s3(database)[Amazon S3 lbdb] in storage
    service gcs(database)[Google Cloud Storage lbdb] in storage
    service azure_blob(database)[Azure Blob or ADLS Pending Range IO] in storage
    service aura(database)[Approved Neo4j HTTPS Origin] in remote
    query:T --> B:file_reader
    query:B --> T:http_reader
    query:R --> L:ladybug
    ladybug:T --> B:lady
    ladybug:R --> L:httpfs
    query:B --> T:azure_io
    file_reader:T --> B:relate
    http_reader:B --> T:secret
    http_reader:L --> R:aura
    query:L --> R:secret
    httpfs:T --> B:s3
    httpfs:R --> L:gcs
    %% @ns-zone-allow runtime data
    %% @ns-zone-allow runtime storage
    %% @ns-zone-allow runtime remote

## Ladybug cloud storage execution design

A Ladybug registration can point directly at a checkpointed `.lbdb` object. The native adapter attaches the object to a private in-memory Ladybug database and runs the graph query in embedded Ladybug. DuckDB consumes the typed batches and evaluates residual SQL, including joins, aggregation, ordering and the final limit.

| Object storage | Native route | Credential |
| --- | --- | --- |
| Amazon S3, `s3://` | Packaged Ladybug HTTPFS performs read-only `ATTACH` against the regional AWS endpoint. | A named DuckDB connection secret of type `s3` scoped to the object or a bucket prefix ending in `/`. Temporary credentials can include a session token. [S3 settings](https://docs.ladybugdb.com/extensions/s3/) |
| Google Cloud Storage, `gs://` or `gcs://` | The same Ladybug adapter uses HTTPFS against `storage.googleapis.com`. | A named connection secret of type `gcs` with scoped GCS interoperability HMAC credentials. Use the same URI scheme in the scope and registration. [GCS settings](https://docs.ladybugdb.com/extensions/gcs/) |
| Azure Blob Storage and ADLS, `az://` and `abfss://` | Core rejects these schemes during registration. Ladybug's Azure extension supports external data scans. The pinned extension lacks the seek, size and positional reads needed to open a native graph. | Azure graph attachment requires upstream random-access support and a passing packaged-engine traversal. [Azure extension](https://docs.ladybugdb.com/extensions/azure/) · [Pinned filesystem implementation](https://github.com/LadybugDB/extensions/blob/3e6d45bba993dbcb0a2b80921b9fa5f544b44bb6/azure/src/file_system/azure_file_system.cpp) |

### Register and query an S3 snapshot

Create the scoped secret on each querying connection. The example assumes a secret named `analytics_graph_s3` and a published graph with the labels and properties shown. [Credential setup](../README.md#query-a-ladybug-snapshot-in-s3-or-google-cloud-storage)

```sql
SET GLOBAL qw_enable_remote_graphs = true;
SET GLOBAL qw_ladybug_config = '{
  "alias": "creator_graph",
  "path": "s3://analytics-graphs/snapshots/creator-20261127.lbdb",
  "secret": "analytics_graph_s3"
}';

SELECT g.creator_id, g.post_id
FROM GRAPH_TABLE (
    creator_graph
    MATCH (c:Creator)-[e:AUTHORED]->(p:Post)
    COLUMNS (c.id AS creator_id, p.id AS post_id)
) AS g
ORDER BY g.creator_id, g.post_id;
```

| Stage | Behavior |
| --- | --- |
| Register | Core checks external access, the remote graph setting, the alias and the object URI. Native resolves the named secret, attaches the snapshot, and discovers its catalog in a read transaction. A failed replacement preserves the previous registration. |
| Bind and admit | Core binds the registered catalog and generation, rechecks access and generation at scan admission, and resolves the executing connection's secret. Cloud scans use the host SQL route. |
| Execute | Each scan owns a private Ladybug connection and attachment. The adapter admits a graph read, executes Cypher against the object, and returns checked batches. The graph file is accessed through signed metadata and range requests. Credentials are passed through the private connection's configuration. |

### Publish and retain a checkpoint

Checkpoint and close the graph before upload. Publish each revision to a new immutable key and retain the old object until its admitted readers finish. Replacing an object in place could let a single reader combine ranges from different versions. Enable external access and allow object reads plus enough metadata access to distinguish an absent WAL from an authorization failure. On S3 a missing key yields 403 when the identity lacks `s3:ListBucket`. The adapter rejects an ambiguous response or a nonempty WAL. The runtime principal needs no object write or delete permissions. [Amazon S3 HeadObject permissions](https://docs.aws.amazon.com/AmazonS3/latest/API/API_HeadObject.html) · [S3 conditional writes](https://docs.aws.amazon.com/AmazonS3/latest/userguide/conditional-writes.html)

The build pins Ladybug 0.23.0, embeds matching HTTPFS and opens attached storage read-only. The engine SQLLogic test exercises the read-only attachment contract. The packaged provider suite checks signed S3 and GCS requests, exact edge rows, SQL residuals, permissions and generation changes. Run `make native-ladybug-engine-test` and `make native-ladybug-cloud-test` with the pinned runner and fixture CLI. [Engine case](../test/ladybug_cloud/engine_attach.test) · [Packaged SQLLogic case](../test/ladybug_cloud/sqllogic/cloud_native_attach.test)

### S3 prerequisite evidence and release checks

An authenticated upstream control queried the same checkpointed, single-edge graph with two Ladybug builds. Stock 0.22.3 scanned nodes and failed the attached edge query. Stock 0.23.0 returned the expected edge, but both builds initiated a multipart upload when attaching the graph from a writable host. Denying `s3:PutObject` prevented stock 0.23.0 from attaching. This motivated the packaged engine's read-only attachment change. [Upstream attachment documentation](https://docs.ladybugdb.com/extensions/attach/lbug/) · [Attached-table edge fix](https://github.com/LadybugDB/ladybug/pull/1493)

The macOS ARM64 Quack-Weave release package ran `GRAPH_TABLE` against a private S3 object in `ap-southeast-1`. A bucket policy denied writes, deletes and multipart aborts during the query. Ordering by edge ID returned `(10, Bob), (11, Bob), (12, Carol), (40, Dina)`. The post-query AWS check found zero pending multipart uploads, and cleanup confirmed bucket deletion. The deterministic fixture separately recorded the HTTP methods. Package SHA-256 `dbe9f8d9eb7f80198a842a423cd2f2c642f9c689968a0881c586596147ed828e`, graph SHA-256 `1a33a3e9160c3f0f2d8bc95d19a5ee738f3ad9261b0829191d90df38bcf98673`. GCS evidence comes from the signed local fixture.

| Verification layer | Reproduce and inspect |
| --- | --- |
| Embedded engine | `make native-ladybug-engine-test` checks the edge oracle and the S3/GCS request methods. It rejects a redirect, an incorrect range and a short body. [Engine SQLLogic witness](../test/ladybug_cloud/engine_attach.test) |
| DuckDB package | `make native-ladybug-cloud-test` checks signed requests, standalone and hybrid queries, duplicates, NULLs, host dispatch, secret scope and revocation, failed replacement and a superseded prepared plan. Its separate stalled-HEAD case checks the configured timeout before reading graph bytes. [Cloud SQLLogic cases](../test/ladybug_cloud/sqllogic) |
| Local graph controls | Run the portable and native unit suites, native SQLLogic stabilization, and two matched Hybrid Query tracker runs for Ladybug and RelateDB. The tracker compares each result against its independent oracle. [Stabilization manifest](../test/stabilization/case_manifest.csv) · [Hybrid test bench](hybrid-test-bench-guide.md) |

Jev interpreted the native admission requirement as `configuration.requires_any`. The pre-change and post-change Z3 models required the scoped connection secret, matching HTTPFS, read-only attached storage and graph-read admission. The validated configurations passed, and removing the scoped secret failed the model. Receipts are `sol_ce8e005d18c44a93` (pre), `sol_8e62556a166b4009` (post) and `sol_118f213a731a4638` (missing secret). The SQLLogic cases exercise those boundaries in the packaged implementation.

## Types crossing the core/native boundary

| Contract | Types and ownership |
|---|---|
| Startup and capability injection | Native constructs `GraphProvider` from `ProviderDescriptor` and an `Arc<dyn ReadBackend>`. `ProviderSource` selects `LocalFile` or `RemoteEndpoint`. `ProviderRewrite` selects `Disabled`, `LabelCatalog`, or `ObservedCatalog`. Core returns a `ProviderRegistry` clone for the native session graph option. |
| Registration and cost evidence | `PreparedRegistration<'a>` returns `GraphCatalog`, an optional single-pattern row bound, optional `GraphPopulations`, and a publication callback. `GraphCatalog` is a map of label or kind-qualified names to typed properties. `GraphPopulations` carries by-table counts or node and edge totals. The callback may return a `RetiredRegistration` to drop outside the registry mutex. Core uses these values for schema, bind-time estimates and constrained rewrites. |
| Admission | Core binds an `Arc<ConfigSnapshot>` with a `GraphGeneration` and the selected backend. Local `open_for_generation` gets the path, schema, timeout and admitted generation. Remote `open_remote` gets the executing `Context`, `RemoteSource`, `ScanLimits` and schema. `RemoteCredential` identifies the credential reference transported to the remote adapter. |
| GQL conversion | `GqlParser` maps source text to `ParsedRead` or `ParseError`. The AST carries `NodePattern`, `Property`, `Projection`, `Predicate`, `PredicateStyle`, `Comparison` and `Literal`. Error diagnostics use `ParseErrorKind`, `Location` and `SourceSpan`. Core owns the supported conversion to Cypher. |
| Scan offers | `ReadBackend::runtime_watch_keys` returns copied `i64` keys. Core represents them as `ExactRuntimeKeys`. `OrderedWatchlistLimit` carries the graph order slot, exact join slot and positive limit. The proof comes from the native C++ optimizer and the Ladybug adapter owns the Cypher ordering suffix. |
| Read request and ownership | `ReadBackend` opens a boxed `ReadSession`. `ReadSession::execute_with_params` consumes the session and accepts Cypher and a slice of named `GraphParameter` values. Supported parameter variants are null, bool, integer, double, and text. The result is a boxed `BatchReader` or a `BackendError`. |
| Results | A `ColumnSpec` holds a name and `ScalarType`. The explicit SQL schema supports BOOLEAN, BIGINT, DOUBLE and VARCHAR. Inferred graph values also use STRUCT and LIST. A `Batch` owns `ColumnData` with a row count. A nested `GraphColumn` owns typed `GraphValue` rows. Core checks the returned schema and batch layout, then writes DuckDB `DataChunk` vectors. |
| Errors and EOF | `BatchReader::next_batch(max_rows, max_bytes)` returns `Some(Batch)` for a valid nonempty batch and `None` for verified EOF. Core translates `BackendError` to stable SQL families for open, admission, execution, schema, Arrow, timeout, batch budget, remote config, auth, transport and response budget. |

The C++ shim uses a borrowed DuckDB factory handle during installation. It copies bounded `int64_t` keys and exposes a `uint64_t` ordered limit during the synchronous `InitWatch` callback. Rust owns the copied keys and the scan-local projection after the callback returns. The C++ optimizer and grammar shim compile against DuckDB's pinned headers. 

[Core graph contracts](../crates/core/src/qw_query.rs#L62) · [Registry and provider descriptors](../crates/core/src/qw_query/providers.rs) · [Core GQL types](../crates/core/src/gql.rs) · [Typed graph values](../crates/core/src/qw_query/graph_value.rs) · [Native graph entry and FFI](../crates/native/src/graph_entry.rs) · [Build setup](../crates/native/build.rs)

## From SQL to the first provider batch

1. `GRAPH_TABLE` lowers into a DuckDB table function and a verified direct-pattern descriptor. Standalone `MATCH` lowers through the selected session graph and dialect. `qw_query` is the explicit Cypher or GQL path with a declared scalar schema and optional named bindings.
2. Core `GraphQuery::bind` resolves the provider and registration snapshot. It validates or infers the schema, binds parameters, and stores a `BoundRead`. Static predicate and projection rewrites use the verified `GraphFragment` and provider capabilities.
3. The optimizer hooks in native C++ attach an eligible runtime join offer and optional ordered-limit proof. A separate pre-optimizer rule can delegate limited ungrouped aggregates under provider and overflow guards. `EXPLAIN` describes available operations using bind information.
4. Core `init_local_state` builds a `ScanProjection`. The first `exec` pull admits the bound generation, opens the provider session, and dispatches the selected request. Later pulls reuse the same reader. Core validates each batch and writes the requested SQL column slots.

This sequence follows a successful local runtime-key read. It models one ordered trace and its declared admission premises.

[Grammar and lowering](../crates/native/grammar/standalone.cpp) · [GRAPH_TABLE](../crates/native/grammar/graph_table.cpp) · [Aggregate rule](../crates/native/grammar/aggregate_pushdown.cpp) · [GraphQuery lifecycle](../crates/core/src/qw_query.rs#L870)

In [ ]:
sequenceDiagram
    participant D as DuckDB
    participant N as Native C++
    participant C as Core GraphQuery
    participant R as Core Registry
    participant A as Native Adapter
    participant G as Graph Engine
    Note over D,G: @spec LOCAL_GRAPH_QUERY<br/>@input current_generation: Bool<br/>@input exact_offer: Bool<br/>@requires PRE: current_generation = true and exact_offer = true
    D->>N: Lower GRAPH_TABLE
    Note over D,N: @message LOWER<br/>@from D<br/>@to N<br/>@event Lower GRAPH_TABLE<br/>@order 1<br/>@when true<br/>@ensures LOWER_CONTRACT: true
    N->>C: Bind table function with PGQ fragment
    Note over N,C: @message BIND<br/>@from N<br/>@to C<br/>@event Bind table function with PGQ fragment<br/>@order 2<br/>@when true<br/>@ensures BIND_CONTRACT: true
    C->>R: Resolve alias snapshot and catalog
    Note over C,R: @message RESOLVE<br/>@from C<br/>@to R<br/>@event Resolve alias snapshot and catalog<br/>@order 3<br/>@when true<br/>@ensures RESOLVE_CONTRACT: current_generation = true
    D->>N: Initialize physical scan with join filter
    Note over D,N: @message INIT<br/>@from D<br/>@to N<br/>@event Initialize physical scan with join filter<br/>@order 4<br/>@when true<br/>@ensures INIT_CONTRACT: true
    N->>C: Expose copied exact-key offer
    Note over N,C: @message OFFER<br/>@from N<br/>@to C<br/>@event Expose copied exact-key offer<br/>@order 5<br/>@when true<br/>@ensures OFFER_CONTRACT: exact_offer = true
    D->>C: Pull graph rows
    Note over D,C: @message PULL<br/>@from D<br/>@to C<br/>@event Pull graph rows<br/>@order 6<br/>@when true<br/>@ensures PULL_CONTRACT: true
    C->>R: Check generation while opening local session
    Note over C,R: @message ADMIT<br/>@from C<br/>@to R<br/>@event Check generation while opening local session<br/>@order 7<br/>@when true<br/>@ensures ADMIT_CONTRACT: current_generation = true
    C->>A: Execute selected read with typed parameters
    Note over C,A: @message REQUEST<br/>@from C<br/>@to A<br/>@event Execute selected read with typed parameters<br/>@order 8<br/>@when true<br/>@ensures REQUEST_CONTRACT: current_generation = true
    A->>G: Invoke provider query
    Note over A,G: @message RUN<br/>@from A<br/>@to G<br/>@event Invoke provider query<br/>@order 9<br/>@when true<br/>@ensures RUN_CONTRACT: true
    A->>C: Return schema and typed batch
    Note over A,C: @message BATCH<br/>@from A<br/>@to C<br/>@event Return schema and typed batch<br/>@order 10<br/>@when true<br/>@ensures BATCH_CONTRACT: true
    C->>D: Write DataChunk for residual SQL
    Note over C,D: @message WRITE<br/>@from C<br/>@to D<br/>@event Write DataChunk for residual SQL<br/>@order 11<br/>@when true<br/>@ensures WRITE_CONTRACT: true
    Note over D,G: @verify LOCAL_PROTOCOL: prove sequence_protocol

### Route selection belongs to scan initialization

The `qw_enable_runtime_watchlist` session switch gates both literal and runtime exact offers. A verified literal candidate takes precedence. The regular runtime branch needs a typed exact key offer. Missing, inexact, or disabled offers keep the host route. Static predicate rewrites have their own eligibility checks. The SQL join still determines the result bag, including duplicate keys and NULLs.

The provider-bound prefix requires `qw_enable_ordered_watchlist_limit`, an eligible Ladybug runtime-key join, the exact join slot, one visible direct BIGINT order value, ASC NULLS LAST, and a positive limit with zero offset. The eligible plan has no additional graph-side filters. DuckDB executes the final join and top-k. Since each matching graph row contributes at least one SQL row and all projected ties carry the same value, the first k qualifying graph rows suffice for this shape. Ladybug supplies the explicit null-last sort expression. RelateDB and literal-key joins use the applicable incumbent route for top-k.

`QuackWeaveWatchlist` logs the exact-key offer or its decline. `QuackWeaveGraphAdmission` reports the acquired generation. `QuackWeaveGraphRoute` logs the selected route once execution returns a reader. An optional prefilter label in `EXPLAIN` describes the available plan and the route log records the dispatch.

[Core route gate](../crates/core/src/qw_query/projection.rs) · [Runtime rules and logging](../crates/core/src/qw_query/runtime_watch.rs) · [Optimizer offer](../crates/native/grammar/runtime_watchlist.cpp) · [Ordered shape proof](../crates/native/grammar/ordered_watchlist.hpp)

In [ ]:
flowchart TD
    S["`@spec ROUTE_AT_SCAN_INIT
@type Status = enum[host,literal_exact,runtime_exact,bounded_exact]
@input enabled: Bool
@input literal_valid: Bool
@input runtime_valid: Bool
@input bound_eligible: Bool
@input bound_enabled: Bool
@input host_sample: Bool
@output status: Status
@requires PRE: true`"]
    B0["`@branch DISPATCH_HOST
@when enabled = false or (literal_valid = false and (runtime_valid = false or host_sample = true))
@ensures DISPATCH_HOST_RESULT: status = host`"]
    B1["`@branch USE_LITERAL
@when enabled = true and literal_valid = true
@ensures USE_LITERAL_RESULT: status = literal_exact`"]
    B2["`@branch USE_BOUND
@when enabled = true and literal_valid = false and runtime_valid = true and host_sample = false and bound_eligible = true and bound_enabled = true
@ensures USE_BOUND_RESULT: status = bounded_exact`"]
    B3["`@branch USE_RUNTIME
@when enabled = true and literal_valid = false and runtime_valid = true and host_sample = false and (bound_eligible = false or bound_enabled = false)
@ensures USE_RUNTIME_RESULT: status = runtime_exact`"]
    V["`@verify COVERED: prove partition_coverage
@verify DISJOINT: prove partition_exclusive
@verify DETERMINISTIC: prove determinism
@verify STATUS_WITNESSES: witness each status`"]
    S --> B0 --> V
    S --> B1 --> V
    S --> B2 --> V
    S --> B3 --> V

## Generation and reader lifecycle

Core publishes a local registration and its provider cache owner under a per-provider write gate. It checks the bound generation and acquires the native session under the matching read gate. Host configuration and logging callbacks stay outside that gate. A replacement that wins before admission causes a stale bound plan to fail with `QW_CONFIG`. An already admitted reader owns the original session resources and can finish after replacement. A freshly rebound query resolves the new registration. Remote read admission uses a final generation check before submission, after no-I/O session construction. S3 and GCS registrations apply this boundary before starting the attached Ladybug session and retains the original object's immutable key for admitted readers.

The first state diagram models the scan cursor. `Dormant` obtains a reader on the first pull. A valid batch keeps it `Active`. Verified EOF or an error releases the reader and finishes the scan. The second state diagram models the local replacement schedules and the invariant that an open session stays pinned to its bound generation.

[Generation gate](../crates/core/src/qw_query/providers.rs#L279) · [Scan state and exec](../crates/core/src/qw_query.rs#L567) · [SQLLogic replacement test](../test/stabilization/sqllogic/usage_binding.test)

In [ ]:
stateDiagram-v2
    [*] --> Dormant
    Dormant --> Active: first_batch
    Dormant --> Finished: empty_eof
    Dormant --> Finished: open_error
    Active --> Active: next_batch
    Active --> Finished: verified_eof
    Active --> Finished: read_error
    note right of Dormant
      @spec SCAN_READER_LIFETIME
      @type Event = enum[first_batch,empty_eof,open_error,next_batch,verified_eof,read_error]
      @input event: Event
      @state-var owns_reader: Bool
      @state-var has_batch: Bool
      @requires PRE: owns_reader = false and has_batch = false
      @state Dormant
      @invariant BATCH_REQUIRES_READER: has_batch = false or owns_reader = true
    end note
    note right of Active
      @state Active
    end note
    note right of Finished
      @state Finished
    end note
    note right of Active
      @transition START
      @from Dormant
      @to Active
      @event event = first_batch
      @guard owns_reader = false
      @update owns_reader' = true
      @update has_batch' = true
    end note
    note right of Finished
      @transition EMPTY
      @from Dormant
      @to Finished
      @event event = empty_eof
      @guard owns_reader = false
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Finished
      @transition START_FAIL
      @from Dormant
      @to Finished
      @event event = open_error
      @guard owns_reader = false
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Active
      @transition NEXT
      @from Active
      @to Active
      @event event = next_batch
      @guard owns_reader = true
      @update has_batch' = true
    end note
    note right of Finished
      @transition END
      @from Active
      @to Finished
      @event event = verified_eof
      @guard owns_reader = true
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Finished
      @transition FAIL
      @from Active
      @to Finished
      @event event = read_error
      @guard owns_reader = true
      @update owns_reader' = false
      @update has_batch' = false
    end note
    note right of Dormant
      @verify INIT_BATCH_REQUIRES_READER: prove initiate BATCH_REQUIRES_READER
      @verify BATCH_REQUIRES_READER_START: prove preserve BATCH_REQUIRES_READER on START
      @verify BATCH_REQUIRES_READER_EMPTY: prove preserve BATCH_REQUIRES_READER on EMPTY
      @verify BATCH_REQUIRES_READER_START_FAIL: prove preserve BATCH_REQUIRES_READER on START_FAIL
      @verify BATCH_REQUIRES_READER_NEXT: prove preserve BATCH_REQUIRES_READER on NEXT
      @verify BATCH_REQUIRES_READER_END: prove preserve BATCH_REQUIRES_READER on END
      @verify BATCH_REQUIRES_READER_FAIL: prove preserve BATCH_REQUIRES_READER on FAIL
    end note

In [ ]:
stateDiagram
    [*] --> Bound
    Bound --> Open: open_current
    Bound --> Stale: replace_before
    Open --> Superseded: replace_after
    Stale --> Done: reject
    Superseded --> Done: drain
    note right of Bound
      @spec LOCAL_GENERATION_ADMISSION
      @type Event = enum[open_current,replace_before,replace_after,reject,drain]
      @input event: Event
      @state-var current: Int
      @state-var bound: Int
      @state-var pinned: Int
      @state-var opened: Bool
      @requires PRE: current = bound and pinned = bound and opened = false
      @state Bound
      @invariant READER_PINS_BOUND: opened = false or pinned = bound
    end note
    note right of Open
      @state Open
    end note
    note right of Stale
      @state Stale
    end note
    note right of Superseded
      @state Superseded
    end note
    note right of Done
      @state Done
    end note
    note right of Open
      @transition ADMIT
      @from Bound
      @to Open
      @event event = open_current
      @guard current = bound and opened = false
      @update opened' = true
      @update pinned' = current
    end note
    note right of Stale
      @transition BEFORE
      @from Bound
      @to Stale
      @event event = replace_before
      @guard current = bound and opened = false
      @update current' = current + 1
    end note
    note right of Superseded
      @transition AFTER
      @from Open
      @to Superseded
      @event event = replace_after
      @guard current = bound and opened = true
      @update current' = current + 1
    end note
    note right of Done
      @transition REJECT
      @from Stale
      @to Done
      @event event = reject
      @guard current > bound and opened = false
      @update opened' = false
    end note
    note right of Done
      @transition DRAIN
      @from Superseded
      @to Done
      @event event = drain
      @guard current > bound and opened = true
      @update opened' = false
    end note
    note right of Bound
      @verify INIT_READER_PINS_BOUND: prove initiate READER_PINS_BOUND
      @verify READER_PINS_BOUND_ADMIT: prove preserve READER_PINS_BOUND on ADMIT
      @verify READER_PINS_BOUND_BEFORE: prove preserve READER_PINS_BOUND on BEFORE
      @verify READER_PINS_BOUND_AFTER: prove preserve READER_PINS_BOUND on AFTER
      @verify READER_PINS_BOUND_REJECT: prove preserve READER_PINS_BOUND on REJECT
      @verify READER_PINS_BOUND_DRAIN: prove preserve READER_PINS_BOUND on DRAIN
    end note

### Dependencies along the read and registration paths

These schedules use one artificial slot per dependency on an arbitrary reference date. They show prerequisites for one read and one registration. Actual timing and overlap come from the runtime and independent provider gates.

In [ ]:
gantt
    title Local read prerequisites on a logical clock
    dateFormat YYYY-MM-DD
    axisFormat %d
    section Dependency order
    Bind snapshot :bind, 2000-01-01, 1d
    Acquire session under admission gate :admit, after bind, 1d
    Submit selected Cypher :request, after admit, 1d
    Validate and publish typed batch :batch, after request, 1d

In [ ]:
timeline
    title Registration ordering with logical slots
    Read registration inputs :prepare, 2000-01-01, 1d
    Validate file identity and descriptor :validate, after prepare, 1d
    Publish registration and backend owner :publish, after validate, 1d
    Admit a bound reader :admit, after publish, 1d

## Evidence flows back into the registration snapshot

`qw_graph_route_feedback` defaults to `off`. A structured runtime-key scan can collect a sample in `shadow` or explicitly run a broad `host_sample` for comparison. Core times active provider and bridge pulls. At verified EOF it checks that the generation is current before publishing. Failed or partial reads produce no sample.

The cache holds one shape per registration, keyed by the base graph query, projected slots, exact key slot and normalized key set, and the verified ordered shape when present. It compares the latest completed exact, host, or bounded samples for that shape. A new registration gets a fresh cache. The shadow output is a comparison for the operator to inspect. Runtime selection continues to follow the route and capability gates above.

[Feedback cache](../crates/core/src/qw_query/route_feedback.rs) · [Scan integration](../crates/core/src/qw_query.rs#L893) · [Feedback SQLLogic](../test/relate/graph_route_feedback.test)

In [ ]:
graph TD
    S["`@spec COMPLETED_SCAN_FEEDBACK
@type Mode = enum[off,shadow,host_sample]
@type Status = enum[ignore,sample_only,report_comparison]
@input mode: Mode
@input has_probe: Bool
@input verified_eof: Bool
@input generation_current: Bool
@input paired_same_shape: Bool
@output status: Status
@requires PRE: true`"]
    B0["`@branch IGNORE
@when mode = off or has_probe = false or verified_eof = false or generation_current = false
@ensures IGNORE_RESULT: status = ignore`"]
    B1["`@branch RECORD
@when mode != off and has_probe = true and verified_eof = true and generation_current = true and (mode = host_sample or paired_same_shape = false)
@ensures RECORD_RESULT: status = sample_only`"]
    B2["`@branch COMPARE
@when mode = shadow and has_probe = true and verified_eof = true and generation_current = true and paired_same_shape = true
@ensures COMPARE_RESULT: status = report_comparison`"]
    V["`@verify COVERED: prove partition_coverage
@verify DISJOINT: prove partition_exclusive
@verify DETERMINISTIC: prove determinism
@verify STATUS_WITNESSES: witness each status`"]
    S --> B0 --> V
    S --> B1 --> V
    S --> B2 --> V

## Observable execution evidence

Run the Arrow input cell below before rerunning the four chart proofs. The cell publishes the fixed observations and the diagrams bind their schema and data hashes.

The two Sankey paths count graph rows delivered before the DuckDB join. They represent separate runs of the local LSQB hot top-16 query with default DuckDB optimizers and the same SQL result. The diagram stops at the scan boundary because a residual join can multiply rows. The two-provider latency chart shows medians from 12 paired runs of the same local query, with ordered limit off and on. The setting changes the Ladybug path on this shape. RelateDB uses its exact route in both arms.

The provider pie counts three adapters in the default native build. The verification chart counts cases in the final core Rust, provider SQLLogic, and stabilization runs. The last two suites passed 655 and 1,183 assertions. These counts and the LSQB numbers describe the local artifact built for `23d96d4`.

[Combined experiment notebook](../quack-weave-core-combined-benchmark.ipynb) · [Ordered-limit SQLLogic](../test/relate/ordered_watchlist_limit.test) · [Shared route SQLLogic](../test/relate/watchlist_applied_route.test)

In [3]:
import pyarrow as pa

# Fixed observations at the documented source snapshot. Values are integer
# ticks recorded as decimal text for the NS-Mermaid Arrow binding.
datasets = [
    {
        "port": "scan_delivery",
        "cols": [
            "source",
            "target",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Exact native read",
                "Exact typed batches",
                "2631"
            ],
            [
                "Exact typed batches",
                "Exact DuckDB scan",
                "2631"
            ],
            [
                "Bounded native read",
                "Bounded typed batches",
                "16"
            ],
            [
                "Bounded typed batches",
                "Bounded DuckDB scan",
                "16"
            ]
        ]
    },
    {
        "port": "lsqb_hot_latency",
        "cols": [
            "category",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Ladybug off",
                "2122"
            ],
            [
                "Ladybug on",
                "1874"
            ],
            [
                "RelateDB off",
                "3560"
            ],
            [
                "RelateDB on",
                "3574"
            ]
        ]
    },
    {
        "port": "provider_inventory",
        "cols": [
            "label",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Local files",
                "2"
            ],
            [
                "Remote HTTPS",
                "1"
            ]
        ]
    },
    {
        "port": "verification_cases",
        "cols": [
            "category",
            "value"
        ],
        "unit": "1",
        "rows": [
            [
                "Core Rust",
                "10"
            ],
            [
                "Provider SQLLogic",
                "22"
            ],
            [
                "Stabilization",
                "30"
            ]
        ]
    }
]

for dataset in datasets:
    schema = pa.schema([
        pa.field(name, pa.string(), nullable=False,
                 metadata={b"unit": dataset["unit"].encode()} if name == "value" else None)
        for name in dataset["cols"]
    ])
    columns = [pa.array(values, type=pa.string()) for values in zip(*dataset["rows"])]
    spur.put(dataset["port"], pa.Table.from_arrays(columns, schema=schema))


source,target,value
Exact native read,Exact typed batches,2631
Exact typed batches,Exact DuckDB scan,2631
Bounded native read,Bounded typed batches,16
Bounded typed batches,Bounded DuckDB scan,16


category,value
Ladybug off,2122
Ladybug on,1874
RelateDB off,3560
RelateDB on,3574


label,value
Local files,2
Remote HTTPS,1


category,value
Core Rust,10
Provider SQLLogic,22
Stabilization,30


In [ ]:
sankey
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=scan_delivery snapshot=7cc2efcce05432d1fc3b3c8b9d8512cc0d75a862576d20e417dbd028b092794d schema=44f729ca7d2a5dc225e92c5d599811db8260be3ae36f104f056c5f54d935e768 unit=1
    Exact native read,Exact typed batches,2631
    Exact typed batches,Exact DuckDB scan,2631
    Bounded native read,Bounded typed batches,16
    Bounded typed batches,Bounded DuckDB scan,16

In [ ]:
xychart
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=lsqb_hot_latency snapshot=50c238b97f9ab55c336ab133b12a25c7c7886a440cfe2d27628b8cfd47950729 schema=f7188d1799d47517ce745717ab86a9aa97dbf6c7375e3bca1198b649a90543a8 unit=1
    title "LSQB hot top-16 paired replay (median microseconds)"
    x-axis ["Ladybug off", "Ladybug on", "RelateDB off", "RelateDB on"]
    y-axis "Microseconds"
    bar [2122, 1874, 3560, 3574]

In [ ]:
pie title Native provider adapters by source
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=provider_inventory snapshot=2a81ac26893086ef6516d7cecd4f87598ffbfa8195b89180c25459e99f63b17a schema=8ee936a11a67759da60dd14c7f971b7ed31831f78b26bbf77ef932016a3f4edd unit=1
    "Local files" : 2
    "Remote HTTPS" : 1
    %% @ns-total 3

In [ ]:
xychart-beta
    %% @ns-bind producer=149feaa5-cbf2-406a-99b0-6b7484b6d5c9 port=verification_cases snapshot=d34c2a8617e4c8cd168c439dde07640f36cadb61635a02b213f81c2f6d07f60e schema=f7188d1799d47517ce745717ab86a9aa97dbf6c7375e3bca1198b649a90543a8 unit=1
    title "Verification suites at 23d96d4"
    x-axis ["Core Rust", "Provider SQLLogic", "Stabilization"]
    y-axis "Test cases"
    bar [10, 22, 30]

## Hybrid Query reproducibility with SQLLogicTest

Hybrid Query here means DuckDB SQL composed with a property-graph read through `GRAPH_TABLE`. DuckDB owns the final join, filters, grouping and ordering. The same logical workload should have one versioned input and an independently established expected result. DuckDB SQLLogicTest should exercise the compiled extension at the public SQL entry point. The existing benchmark tracker should consume the same workload identity and expected result at benchmark scale.

### Practices to adopt

| Source | Practice for Quack-Weave |
|---|---|
| SQLite and DuckDB SQLLogicTest | Use small declarative files with setup, query results and expected errors. Assert ordered rows with a total visible order when order is part of the contract. Use `rowsort` for unordered row bags and shared query labels for equivalent forms. Check schema explicitly when it matters, since DuckDB's query type letters are legacy column-count markers. [SQLite format](https://sqlite.org/sqllogictest/doc/trunk/about.wiki) · [DuckDB result verification](https://duckdb.org/docs/2.0/dev/sqllogictest/result_verification) |
| Apache DataFusion | Give cases isolated scratch space and make expected-output regeneration an explicit maintainer operation. In Quack-Weave, generate large goldens from a relational oracle and review the diff. [DataFusion runner](https://github.com/apache/datafusion/blob/main/datafusion/sqllogictest/README.md) |
| DuckDB benchmarks and LDBC SNB | Separate setup, execution and validation. Pin the query sequence and inputs for replay. Keep release timing distinct from diagnostic execution. LDBC likewise defines separate driver modes for sequential cross-validation and concurrent measurement. [DuckDB benchmark suite](https://duckdb.org/docs/2.0/dev/benchmark) · [LDBC SNB specification, §7.4.2](https://ldbc.github.io/ldbc_snb_docs/ldbc-snb-specification.pdf#page=83) |
| LSQB and MICRO CM-LDBC | Reuse LSQB for graph patterns, fanout, and dataset scale. Add relational predicates, joins on both graph endpoints, and grouping from the CM-LDBC workload pattern. The current tracker already includes local adaptations of those shapes. [LSQB](https://github.com/ldbc/lsqb) · [MICRO technical report, §VII-B](https://arxiv.org/html/2603.13835v1#S7.SS2) |
| NIST combinatorial testing | Cover legal provider, query-shape, route and data combinations with a constrained pairwise matrix as the suite grows. Keep explicit higher-order cases where guards interact. [NIST SP 800-142](https://csrc.nist.gov/pubs/sp/800/142/final) |

### Workload and oracle

- Give each case a stable ID and revision. Reference the canonical SQL and ordered typed parameter inputs, a fixture ID and content digests, an independent oracle revision, output schema and comparison mode, and the eligible provider/route variants. The same record can point to a tiny SQLLogic witness and a scaled workload using the same SQL template at different fixture sizes. Assign different instance identities to different scales or parameter sequences.
- Materialize the graph and relational tables from the same immutable source data. Keep small fixtures as explicit rows or reviewable fixture scripts. At scale pin the dataset bytes or the generator version, seed, arguments, stable IDs and ordering, and logical data digests. Record provider materialization and extension/host binary digests separately. Each test gets private state and verifies that read-only provider fixtures were preserved. Pin time zone, collation, relevant session settings and thread mode. Use fixed timestamps and explicit seed inputs where needed. [DuckDB non-deterministic behavior](https://duckdb.org/docs/current/operations_manual/non-deterministic_behavior)
- Write the reference as ordinary SQL over vertex, edge and attribute tables, with explicit SQL NULL behavior, edge direction and path semantics. Check small goldens by hand. Compare the host and optimized routes to the same reference, with a two-way `EXCEPT ALL` check for bag outputs. Check column names and logical types separately. Use exact integer, DECIMAL or fixed-scale projections where exact equality is required. Cases that expose floating point need a declared comparator or canonical representation. Preserve NULLs, empty strings, row boundaries and duplicate multiplicities in the result encoding.
- Use explicit `ORDER BY` and stable visible tie breakers for observable top-k or pagination. The current Ladybug bounded route has a single visible order value, so its positive case should preserve that shape and assert the output values and their multiplicity. Use a separate negative case for a richer projected tie breaker and assert fallback. An optimizer plan offer and a dispatched route are different observations. Assert dispatch with `QuackWeaveGraphRoute` and use a narrow plan or scan-row assertion where the bound matters.
- Keep full expected rows for small cases. For large exact outputs, use a versioned typed encoding, logical schema, row count and SHA-256. Canonicalize bags by sorting intact rows and keep the returned sequence for ordered results. A mismatch should expose sample differing rows or enable an exact rerun. Keep golden updates explicit and sourced from the reference.

[Public SQLLogic cases](../test/stabilization/sqllogic/hybrid_public.test) · [Applied route cases](../test/relate/watchlist_applied_route.test) · [Ordered limit cases](../test/relate/ordered_watchlist_limit.test)

### Scale the suite through the existing runners

| Profile | Inputs and checks |
|---|---|
| PR | Pinned tiny fixtures and explicit rows, with host versus eligible optimized routes. Include duplicate and NULL keys, no-match and empty inputs, parallel edges, and top-k boundary cases. Exercise both local providers where the shape is supported. |
| Nightly | Pinned generated fixtures and an independent exact oracle. Vary provider, join/projection shape, filter placement, key cardinality, thread mode, and scale. Add explicit combinations around the live key cap (below, at, above), inexact offers, route gates, and registration replacement. Synchronize concurrency checks on observed phases. |
| Benchmark | Pinned LSQB projection and CM-LDBC-style inputs. Run exact preflight against the oracle and verify each fetched sample after stopping the timer. Compare the incumbent and candidate on the same workload, with the same schedule and matched runner settings. |
| Remote integration | Use a controlled Neo4j fixture, a pinned server/client version and exact expected rows. Keep external/live endpoint probes under their own CI job and evidence. |

Keep `test/stabilization/run_sqllogictest.py` and its case manifest for public SQL contracts, and `test/relate/run.py --case` for provider dispatch and file-immutability checks. Put small variant loops in SQLLogic and generate larger pinned matrices into ordinary SQLLogic records. The test report should identify the case and variant, expected and observed result or error, artifact and fixture digests, route evidence when applicable, and the inputs needed for a focused replay. Mark unsupported provider/route combinations explicitly and require every selected supported combination to report a result. [Coverage manifest](../test/stabilization/case_manifest.csv) · [Provider runner](../test/relate/run.py) · [DuckDB loops](https://duckdb.org/docs/2.0/dev/sqllogictest/loops)

The current tracker checks fixture hashes, compares per-execution signatures to an independent CSV-derived oracle, and runs dedicated SQLLogic result and snapshot gates in `--micro` mode. Its SF1 signature path currently targets integer-like columns. Extend the shared workload contract first, then add a typed, mode-aware encoder when a workload needs richer types or ordered hashes. [Tracker](../benchmarks/provider_comparison/tracker.py) · [CM-LDBC benchmark SQLLogic](../benchmarks/provider_comparison/tracker/micro_results.test) · [Snapshot SQLLogic](../benchmarks/provider_comparison/tracker/micro_snapshot.test)

### Benchmark replay and comparison

- Reuse the case SQL, typed parameters and expected result. Save a workload fingerprint from the case revision, SQL, fixtures, parameter sequence and oracle. Save an execution fingerprint from the workload fingerprint plus host, extension, provider build and session settings. Across commits, hold the workload and measurement protocol fixed and record the intentional artifact changes.
- Use matched, alternating blocks on the same runner. Capture connection/setup, first query and warm-query timings independently. Time submission through complete client fetch. Keep fixture preparation, result comparison and diagnostic `EXPLAIN ANALYZE` outside that interval. Name a fresh-process run with an already warmed filesystem cache accordingly.
- Record the exact order and seed, warmup and sample counts, all sample times, p50/p95 and sample units, host/CPU/thread settings, timeouts and failures, and the fixture and binary hashes. Preserve route and graph-row diagnostics from a separate execution. Choose enough blocks before treating tail comparisons as actionable. Exact result digests and equivalent input schedules should replay identically. Latency remains a distribution.
- Start with the airport-directory cases and give `micro_creator_selective` its own case record. Add a tiny creator fixture that executes the same SQL template as the scaled creator workload. Check the tiny graph/SQL result twice with independently created scratch state. At scale, require every result digest to equal the oracle and repeat the same schedule with the same inputs. A deliberate dropped duplicate or incorrect LIMIT placement should fail correctness before a snapshot is published. Expand to broad, message-side and dual-endpoint cases once that path has evidence.

[Tracker measurement protocol](../benchmarks/provider_comparison/tracker/README.md#measurement) · [Benchmark workflow](../.github/workflows/Benchmark.yml)

### Jev, Z3 and local replay evidence

| Check | Observed result |
|---|---|
| Suite profile policy | `jev_compile` 1.14.0 opened the gate at 0.99 weakest confidence and selected `data_integrity.mutually_consistent` in verify mode. The bindings matched the proposed four profile/fixture/oracle/measurement tuples. Z3 accepted the proposed rows (`sat`, `sol_53c45fde611140c9`) and rejected the same rows with measurement enabled for PR (`unsat`, `sol_65ed4d6bdc3c4eac`). |
| Bounded LIMIT cardinality | The typed counterexample query covered `1 <= k <= 64`. If the prefix keeps the whole stream when it has fewer than `k` graph rows, and each retained graph row emits at least one residual-join row, the prefix cannot return fewer rows after LIMIT than the full join (`unsat`, `sol_28ec640525b14b29`). Removing the per-row match condition gives a witness with `k=1`, one retained graph row, zero prefix join rows and one host join row (`sat`, `sol_951c6da66ecb4f2d`). The model checks cardinality. SQLLogic checks the actual row values, order and fallback decisions. |
| Tiny public and provider cases | Two fresh-scratch replays each passed 104 SQLLogic records across `hybrid_public.test` (48), `watchlist_applied_route.test` (23) and `ordered_watchlist_limit.test` (33), with DuckDB v2.0.1 and the local extension identified below. |
| Scaled benchmark case | Two tracker runs of `micro_creator_selective` on the same LSQB SF1 `mod32` projection each verified eight warmups and 16 measured calls across two providers and two routes. The fixture, binary and oracle signatures, statements, schedule and SQLLogic gate checks matched. The four-row oracle digest was `7f2df2fb87fce9818960fee6a121a32ab86c7977de5536c33615fbb9b617fe04`. Both runs observed 734 scan rows on the optimized route and 119,052 on the host route for both providers. Latencies varied. These sample counts exercise the replay protocol. |

The smoke runs used a prebuilt local extension with SHA-256 `c662638b1fb0dd8459989c916a8f99e6ae172604761f032e16ecd4af75dba176`. The two focused commands below use a compatible DuckDB SQLLogic runner and extension. The tracker invocation includes both providers and validates the result and snapshot in SQLLogic before writing an output file. Use a new output path for each replay.

```sh
python3 -B test/relate/run.py --duckdb-unittest "$DUCKDB_UNITTEST" \
  --fixture-builder "$QW_RELATE_FIXTURE" --extension "$QW_EXT" --lbug "$LBUG" \
  --case watchlist_applied_route.test --case ordered_watchlist_limit.test

"$TRACKER_PYTHON" -B benchmarks/provider_comparison/tracker.py run \
  --micro --case micro_creator_selective --fixture build/perf/provider-comparison/mod32 \
  --duckdb-unittest "$DUCKDB_UNITTEST" --library "$DUCKDB_LIB" --extension "$QW_EXT" \
  --blocks 2 --runs 2 --out "$QW_BENCHMARK_OUTPUT"
```

## Source map by component

| Component | Types and records to inspect |
|---|---|
| Core request and result | `GraphQuery`, `GraphQueryKind`, `BoundRead`, `GraphLanguage`, `GraphFragment`, `ScanProjection`, `Scan`, `ColumnSpec`, `ScalarType`, `Batch`, `ColumnData`, `GraphColumn` and `GraphValue`. |
| Core registration and policy | `ProviderRegistry`, `RegistryState`, `ProviderSlot`, `GraphConfig`, `ConfigSnapshot`, `RegisteredConfig`, `GraphGeneration`, `RuntimeWatch`, `ExactRuntimeKeys`, `WatchlistRoute`, and feedback `Mode`, `Shape`, `Observation` and `Feedback`. |
| Native C++ | `GraphGrammar` and `GraphConnections` install and activate syntax. `WatchInit`, `KeyOffer`, `KeyCopy` and `QwOrderedWatchlistOffer` hold the temporary plan and scan offers. The aggregate and explain hooks consume graph bind information. |
| Native Rust Ladybug | `LadybugBackend`, `DatabaseCache`, `Session` and `Reader`. Native resources are wrapped by `Database`, `Connection`, `PreparedStatement` and `QueryResult`. |
| Native Rust RelateDB | `RelateBackend`, `RegisteredDatabase`, `Session` and `Rows`. The adapter owns observed type checks and conversion of the eager result. |
| Native Rust Neo4j | `Neo4jBackend`, `Endpoint`, `Session` and the decoded row reader. The secret bridge and adapter own scoped authentication and transport. |

Source navigation for an adapter addition starts at the descriptor in the native entrypoint, follows the injected traits in core, and checks the C++ rules only for features that require a DuckDB plan proof.

[Core source](../crates/core/src) · [Native Rust source](../crates/native/src) · [Native C++ grammar](../crates/native/grammar)

## NS-Mermaid profile coverage

| Registry profile | Mermaid headers | Evidence in this notebook |
|---|---|---|
| Architecture policy | `architecture-beta`, `architecture` | Runtime component and deployment edges |
| Branch relation | `flowchart`, `graph` | Route choice and feedback eligibility |
| State invariant | `stateDiagram-v2`, `stateDiagram` | Cursor and local generation invariants |
| Sequence trace | `sequenceDiagram` | Declared successful local graph read |
| Schedule feasibility | `gantt`, `timeline` | Modeled read and registration prerequisite order |
| Flow conservation | `sankey` | Counts delivered at the graph scan boundary |
| Numeric series | `pie`, `xychart`, `xychart-beta` | Provider inventory, observed latency and verification counts |
| Other numeric headers | `radar`, `quadrant`, `treemap` | Registry reports their series adapters as unavailable |
| Finite sets, workflow, and wire layout | `venn`, `kanban`, `packet`, `packet-beta` | Registry reports finite-set, workflow, and packet lowering as unavailable |
| Bounded relations and Git policy | `classDiagram`, `erDiagram`, `requirementDiagram`, `gitGraph` | Mermaid visual references below |
| Conceptual diagrams | `mindmap`, `journey`, `ishikawa`, `wardley`, `cynefin` | Registry classifies these as visualization-only |

The registry also lists bounded reachability, transition BMC, packet encoding, event trace, and schedule optimization without active Mermaid headers. The executable cells use version 1 of the installed profiles. Mermaid fences in the appendix are visual references for a compatible renderer.

### Rust trait and data model

```mermaid
classDiagram
    class GraphProvider
    class ProviderDescriptor
    class ReadBackend {
        <<interface>>
        prepare_registration()
        open_for_generation()
        open_remote()
    }
    class ReadSession {
        <<interface>>
        execute_with_params()
    }
    class BatchReader {
        <<interface>>
        schema()
        next_batch()
    }
    class GraphParameter
    class OrderedWatchlistLimit
    class Batch {
        len
        columns
    }
    class ColumnSpec
    class ScalarType
    class ColumnData
    class GraphColumn
    class GraphValue
    class LadybugBackend
    class RelateBackend
    class Neo4jBackend
    GraphProvider *-- ProviderDescriptor
    GraphProvider o-- ReadBackend
    LadybugBackend ..|> ReadBackend
    RelateBackend ..|> ReadBackend
    Neo4jBackend ..|> ReadBackend
    ReadBackend ..> ReadSession : opens
    ReadBackend ..> OrderedWatchlistLimit : offers
    ReadSession ..> GraphParameter : consumes named values
    ReadSession ..> BatchReader : transfers read ownership
    BatchReader ..> Batch : yields
    BatchReader ..> ColumnSpec : provides schema
    ColumnSpec *-- ScalarType
    Batch *-- ColumnData
    ColumnData o-- GraphColumn
    GraphColumn o-- ScalarType
    GraphColumn *-- GraphValue
```

### Ownership and requirements

The entity view describes in-memory ownership. It groups one active registration per provider together with the bound plans and feedback it can serve.

```mermaid
erDiagram
    PROVIDER_SLOT ||--o| CONFIG_SNAPSHOT : publishes
    CONFIG_SNAPSHOT ||--|| GRAPH_GENERATION : identifies
    CONFIG_SNAPSHOT ||--o{ BOUND_READ : pins
    CONFIG_SNAPSHOT ||--o| FEEDBACK_ENTRY : caches
    BOUND_READ ||--o{ SCAN_LOCAL_STATE : initializes
    SCAN_LOCAL_STATE ||--|| SCAN_PROJECTION : maps
    SCAN_LOCAL_STATE ||--|| SCAN : tracks
    SCAN ||--o| BATCH_READER : retains
    BATCH_READER ||--o{ TYPED_BATCH : yields
```

```mermaid
requirementDiagram
    requirement same_sql_bag {
        id: 1
        text: "Exact-key routes preserve the SQL result bag"
        risk: high
        verifymethod: test
    }
    requirement admitted_generation {
        id: 2
        text: "An admitted read stays on its bound generation"
        risk: high
        verifymethod: test
    }
    element watchlist_logic {
        type: SQLLogic
    }
    element usage_binding {
        type: SQLLogic
    }
    watchlist_logic - verifies -> same_sql_bag
    usage_binding - verifies -> admitted_generation
```

### Implementation history

```mermaid
gitGraph
    commit id: "54b2db6" tag: "baseline"
    commit id: "29b024c" tag: "shared route gate"
    commit id: "2d3cfec" tag: "generation admission"
    commit id: "c2356f3" tag: "ordered Ladybug prefix"
    commit id: "23d96d4" tag: "shadow feedback"
```

```mermaid
mindmap
    root((Core native handoff))
        Enter through DuckDB
            Native grammar
            Core table function
        Open a graph
            Registry generation
            Native adapter session
        Return typed rows
            BatchReader
            DuckDB DataChunk
        Observe execution
            Offer and dispatch logs
            Completed scan feedback
```